# 03. Feature Engineering

> **Краткое резюме**: Объединяем профильные и поведенческие данные в единую матрицу признаков. Encoding категориальных переменных, log-трансформация, нормализация, обработка пропусков. Результат — `data/feature_matrix.parquet`.

**Предпосылки**: `01_etl_profile` и `02_etl_behavior` выполнены.

---

In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

OUTPUT_DIR = os.path.abspath("./data")
# Spark для чтения Parquet-директорий
try:
    _ = spark
except NameError:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.appName("LookAlike_FE").enableHiveSupport().getOrCreate()

SPARK_DIR = f"file://{OUTPUT_DIR}"

---
## 1. Загрузка данных

In [ ]:
profiles_sdf = spark.read.parquet(f"{SPARK_DIR}/client_profiles.parquet")
behavior_sdf = spark.read.parquet(f"{SPARK_DIR}/behavioral_features.parquet")

# INNER JOIN: оставляем только клиентов у которых есть и профиль и поведение.
# .count() убран — toPandas() сам материализует JOIN, двойной проход не нужен.
merged_sdf = profiles_sdf.join(behavior_sdf, on="client_uk", how="inner")

df = merged_sdf.toPandas()
df = df.set_index("client_uk")
print(f"Loaded to Pandas: {df.shape}")
df.head()

---
## 2. Производные признаки

In [ ]:
## 2b. Граф-метрики (опционально)
# Если основной пайплайн (src/pipeline.py) был запущен и сохранил graph_metrics.parquet,
# присоединяем PageRank, betweenness, clustering_coef, flow_through_ratio, role.
# Это самые ценные признаки для look-alike: позиция клиента в сети транзакций.
#
# Чтобы сгенерировать файл, добавь в конец run_analysis_pipeline():
#   metrics_df.to_parquet(os.path.join(data_dir, "graph_metrics.parquet"))

GRAPH_METRICS_PATH = os.path.join(OUTPUT_DIR, "graph_metrics.parquet")
GRAPH_METRIC_COLS = [
    "pagerank",  # Взвешенный PageRank: влиятельность в сети
    "betweenness",  # Центральность по посредничеству: «мост» между группами
    "clustering_coef",  # Плотность локального кластера клиента
    "flow_through_ratio",  # Транзитный клиент (высокий = деньги «проходят насквозь»)
    "in_degree",  # Число входящих связей
    "out_degree",  # Число исходящих связей
    "top_k_concentration",  # Концентрация оборота на топ-K контрагентов
    "okved_diversity_count",  # Отраслевая диверсификация контрагентов
    "okved_diversity_entropy",  # Энтропия отраслевого микса
]

if os.path.exists(GRAPH_METRICS_PATH):
    gm = pd.read_parquet(GRAPH_METRICS_PATH)
    available_gm_cols = [c for c in GRAPH_METRIC_COLS if c in gm.columns]
    # role → one-hot encoding
    if "role" in gm.columns:
        role_dummies = pd.get_dummies(gm["role"].fillna("regular"), prefix="role")
        gm = pd.concat([gm[available_gm_cols], role_dummies], axis=1)
        available_gm_cols = list(gm.columns)
    # JOIN по client_uk (index)
    gm.index.name = "client_uk"
    df = df.join(gm[available_gm_cols], how="left")
    df[available_gm_cols] = df[available_gm_cols].fillna(0)
    print(f"✅ Graph metrics loaded: {available_gm_cols}")
else:
    available_gm_cols = []
    print(
        f"⚠️  graph_metrics.parquet not found at {GRAPH_METRICS_PATH}\n"
        "   Run the main pipeline and add:\n"
        "   metrics_df.to_parquet(os.path.join(data_dir, 'graph_metrics.parquet'))\n"
        "   Graph-based features will be skipped."
    )

In [ ]:
# Direction ratio: outflow / total (0 = только получает, 1 = только платит)
df["direction_ratio"] = df["total_outflow"] / df["total_amount"].replace(0, np.nan)
df["direction_ratio"] = df["direction_ratio"].fillna(0.5)

# Средний месячный оборот
df["avg_monthly_amount"] = df["total_amount"] / df["active_months"].replace(0, 1)

# Тренд оборота: (вторая половина - первая) / первая
df["amount_growth"] = (
    (
        (df["amount_second_half"] - df["amount_first_half"])
        / df["amount_first_half"].replace(0, np.nan)
    )
    .clip(-5, 5)
    .fillna(0)
)

# Рост контрагентов
df["cp_growth"] = (
    ((df["cp_second_half"] - df["cp_first_half"]) / df["cp_first_half"].replace(0, np.nan))
    .clip(-5, 5)
    .fillna(0)
)

# Коэффициент вариации суммы транзакции
df["tx_amount_cv"] = df["std_tx_amount"] / df["avg_tx_amount"].replace(0, np.nan)
df["tx_amount_cv"] = df["tx_amount_cv"].fillna(0)

# ── Интенсивность (нормированные по времени) ─────────────────────────────────
# Вместо абсолютных count-значений показывают поведение в единицу времени.
# tx_per_month = 5 транзакций/мес vs tx_count = 15 при 3 месяцах — это разные клиенты.
df["tx_per_month"] = df["tx_count"] / df["active_months"].replace(0, 1)
df["cp_per_month"] = df["unique_counterparties"] / df["active_months"].replace(0, 1)

# ── Структура контрагентов ────────────────────────────────────────────────────
# payee_ratio: доля уникальных получателей среди всех контрагентов (платящий клиент)
# payer_ratio: доля уникальных плательщиков (клиент-получатель)
df["payee_ratio"] = (
    (df["unique_payees"] / df["unique_counterparties"].replace(0, np.nan)).fillna(0.5).clip(0, 1)
)

df["payer_ratio"] = (
    (df["unique_payers"] / df["unique_counterparties"].replace(0, np.nan)).fillna(0.5).clip(0, 1)
)

# ── Концентрация оборота ──────────────────────────────────────────────────────
# Средний оборот на контрагента: высокий = несколько крупных партнёров,
# низкий = много мелких. Рассчитывается до log1p.
if "total_outflow" in df.columns:
    df["amt_per_cp"] = (
        df["total_outflow"] / df["unique_counterparties"].replace(0, np.nan)
    ).fillna(0)
    df["amt_per_cp"] = df["amt_per_cp"].clip(lower=0)

print("Derived features created.")
new_cols = ["tx_per_month", "cp_per_month", "payee_ratio", "payer_ratio"]
if "amt_per_cp" in df.columns:
    new_cols.append("amt_per_cp")
print(df[new_cols].describe().round(3))

---
## 3. Encoding категориальных признаков

In [ ]:
# ОКВЭД: top-N + 'OTHER'
OKVED_TOP_N = 20
if "sparkokved_ccode" in df.columns:
    okved = df["sparkokved_ccode"].fillna("UNKNOWN")
    top_okveds = okved.value_counts().head(OKVED_TOP_N).index.tolist()
    okved_mapped = okved.where(okved.isin(top_okveds), "OTHER")
    okved_dummies = pd.get_dummies(okved_mapped, prefix="okved")
    df = pd.concat([df, okved_dummies], axis=1)
    print(f"OKVED: {OKVED_TOP_N} top codes + OTHER → {okved_dummies.shape[1]} dummy columns")

# Регион: top-N + 'OTHER'
REGION_TOP_N = 20
if "addrref_region_uk" in df.columns:
    region = df["addrref_region_uk"].fillna(-1).astype(int).astype(str)
    top_regions = region.value_counts().head(REGION_TOP_N).index.tolist()
    region_mapped = region.where(region.isin(top_regions), "OTHER")
    region_dummies = pd.get_dummies(region_mapped, prefix="region")
    df = pd.concat([df, region_dummies], axis=1)
    print(f"Region: {REGION_TOP_N} top + OTHER → {region_dummies.shape[1]} dummy columns")

# Модельный сегмент (если доступен)
if "srvpackagesegment_ccode" in df.columns:
    seg = df["srvpackagesegment_ccode"].fillna("UNKNOWN")
    seg_dummies = pd.get_dummies(seg, prefix="mseg")
    df = pd.concat([df, seg_dummies], axis=1)
    print(f"Model segment → {seg_dummies.shape[1]} dummy columns")
else:
    print("Model segment not available — skipped")

# Тип клиента
if "client_type_name" in df.columns:
    ctype_dummies = pd.get_dummies(df["client_type_name"].fillna("UNKNOWN"), prefix="ctype")
    df = pd.concat([df, ctype_dummies], axis=1)
    print(f"Client type → {ctype_dummies.shape[1]} dummy columns")

---
## 4. Log-трансформация и нормализация

In [ ]:
# Формируем финальный набор числовых фичей
NUMERIC_FEATURES = [
    # Поведенческие (абсолютные)
    "total_amount",
    "total_outflow",
    "total_inflow",
    "tx_count",
    "avg_tx_amount",
    "std_tx_amount",
    "unique_counterparties",
    "unique_payees",
    "unique_payers",
    "active_months",
    # Производные (доли, темпы, коэффициенты)
    "direction_ratio",
    "avg_monthly_amount",
    "amount_growth",
    "cp_growth",
    "tx_amount_cv",
    # Интенсивность (нормированные по времени)
    "tx_per_month",
    "cp_per_month",
    # Структура контрагентов
    "payee_ratio",
    "payer_ratio",
    # Концентрация оборота
    "amt_per_cp",
    # Граф-метрики (если доступны)
    *[c for c in available_gm_cols if not c.startswith("role_")],
]
# Опциональные (зависят от источника данных)
for col in [
    "avg_balance",
    "max_balance",
    "min_balance",
    "avg_balance_30d",
    "product_count",
    "product_type_count",
    "product_total_amt",
    "registry_authcapital_amt",
    "revenue_rur_amt",
]:
    if col in df.columns:
        NUMERIC_FEATURES.append(col)

# Dummy-колонки (ОКВЭД, регион, сегмент, тип, роль из графа)
DUMMY_COLS = [
    c for c in df.columns if c.startswith(("okved_", "region_", "mseg_", "ctype_", "role_"))
]

FEATURE_COLS = [c for c in NUMERIC_FEATURES + DUMMY_COLS if c in df.columns]

n_num = len([c for c in NUMERIC_FEATURES if c in df.columns])
n_gm = len([c for c in available_gm_cols if c in df.columns])
print(f"Feature columns: {len(FEATURE_COLS)} ({n_num} numeric + {len(DUMMY_COLS)} dummy)")
if n_gm:
    print(f"  incl. {n_gm} graph-metric features")

In [ ]:
# Формируем финальный набор числовых фичей
NUMERIC_FEATURES = [
    # Поведенческие (абсолютные)
    "total_amount",
    "total_outflow",
    "total_inflow",
    "tx_count",
    "avg_tx_amount",
    "std_tx_amount",
    "unique_counterparties",
    "unique_payees",
    "unique_payers",
    "active_months",
    # Производные (доли, темпы, коэффициенты)
    "direction_ratio",
    "avg_monthly_amount",
    "amount_growth",
    "cp_growth",
    "tx_amount_cv",
    # Интенсивность (нормированные по времени)
    "tx_per_month",
    "cp_per_month",
    # Структура контрагентов
    "payee_ratio",
    "payer_ratio",
    # Концентрация оборота
    "amt_per_cp",
]
# Опциональные (зависят от источника данных)
for col in [
    "avg_balance",
    "max_balance",
    "min_balance",
    "avg_balance_30d",
    "product_count",
    "product_type_count",
    "product_total_amt",
    "registry_authcapital_amt",
    "revenue_rur_amt",
]:
    if col in df.columns:
        NUMERIC_FEATURES.append(col)

# Dummy-колонки (ОКВЭД, регион, сегмент, тип)
DUMMY_COLS = [c for c in df.columns if c.startswith(("okved_", "region_", "mseg_", "ctype_"))]

FEATURE_COLS = [c for c in NUMERIC_FEATURES + DUMMY_COLS if c in df.columns]

n_num = len([c for c in NUMERIC_FEATURES if c in df.columns])
print(f"Feature columns: {len(FEATURE_COLS)} ({n_num} numeric + {len(DUMMY_COLS)} dummy)")

In [ ]:
# Заполнение пропусков и нормализация
X = df[FEATURE_COLS].fillna(0).copy()

# StandardScaler для числовых
numeric_in_X = [c for c in NUMERIC_FEATURES if c in X.columns]
scaler = StandardScaler()
X[numeric_in_X] = scaler.fit_transform(X[numeric_in_X])

print(f"Feature matrix: {X.shape}")
X.describe()

---
## 5. Сохранение

In [ ]:
import pickle

# Сохраняем матрицу фичей
feature_path = os.path.join(OUTPUT_DIR, "feature_matrix.parquet")
X.to_parquet(feature_path)
print(f"Feature matrix saved: {feature_path}")

# Сохраняем метаданные (scaler, список фичей, маппинг ОКВЭД)
meta = {
    "feature_cols": FEATURE_COLS,
    "numeric_features": numeric_in_X,
    "dummy_cols": DUMMY_COLS,
    "scaler": scaler,
}
meta_path = os.path.join(OUTPUT_DIR, "feature_meta.pkl")
with open(meta_path, "wb") as f:
    pickle.dump(meta, f)
print(f"Metadata saved: {meta_path}")

# Сохраняем полный df (с сырыми + derived + dummy колонками) для анализа
full_path = os.path.join(OUTPUT_DIR, "full_client_data.parquet")
# Сохраняем только сериализуемые колонки
save_cols = [
    c
    for c in df.columns
    if df[c].dtype.name != "object"
    or c
    in [
        "client_name",
        "client_type_name",
        "clientcounterparty_flag",
        "sparkokved_ccode",
        "addrref_city_name",
        "reg_city_name",
        "srvpackagesegment_ccode",
        "clientchange_date",
        "client_rel_date",
        "entrepreneur_flag",
        "client_active_flag",
    ]
]
df[save_cols].to_parquet(full_path)
print(f"Full data saved: {full_path}")

---

## Глоссарий

| Термин | Описание |
|--------|----------|
| **direction_ratio** | Доля исходящих платежей: 0 = только получает, 1 = только платит |
| **amount_growth** | Рост оборота: (2-я половина - 1-я) / 1-я. Clipped [-5, 5] |
| **cp_growth** | Рост числа контрагентов по той же формуле |
| **tx_amount_cv** | Коэффициент вариации суммы транзакции (std / mean). Высокий = неравномерные платежи |
| **log1p** | Логарифм (1+x). Сжимает распределение монетарных переменных |
| **StandardScaler** | Нормализация к среднему=0, std=1. Необходима для K-Means и distance-based методов |
| **One-hot (dummy)** | Кодирование категориальной переменной: 1 колонка на каждое значение |

---

**Следующий шаг**: `04_segmentation.ipynb`.